# SplatForge A5 — LoRA fine-tune on an A100 (Colab)

Trains a **real PEFT/LoRA adapter** for the grasp policy on the robot's **own**
rollouts. The base net aims at nominal (0,0); the adapter learns to localise the
mug from a noisy observation, so the grasp lands. "The model updates its own
weights from self-generated experience."

Runtime: **Runtime → Change runtime type → A100 GPU**. No API keys needed for
training (Gemini is only for the curriculum/critic when re-banking the curve).

## 0. GPU check

In [ ]:
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available())

## 1. Clone + install
Uses the A5 branch until it's merged; after merge, set `BRANCH = 'main'`.

In [ ]:
BRANCH = 'shivamsinghnow/spl-11-a5-lora-finetune-harness'
!git clone -q https://github.com/ShivamSinghNow/SplatForge.git
%cd SplatForge
!git checkout -q $BRANCH
!pip -q install -e . peft

## 2. Generate the robot's own rollout data
Run the scripted policy across many randomized scenes; each rollout's mug
position becomes a training example (observed through sensor noise).

In [ ]:
import tempfile
from splatforge.scanning.scenes import load_scene
from splatforge.sim.task import build_pick_task
from splatforge.simulation.mujoco_sim import MujocoSimulationBackend
from splatforge.policy import DEFAULT_POLICY
from splatforge.orchestrator import ReplayBuffer, evaluate_policy, load_local_trajectories
from splatforge.storage import JsonlRepository
from splatforge.policy.lora_trainer import build_training_pairs

scene = load_scene('mug_table')
task = build_pick_task('pick_up_mug')
backend = MujocoSimulationBackend()

data_dir = tempfile.mkdtemp()
evaluate_policy(scene, task, DEFAULT_POLICY, backend, n_rollouts=400, seed=0,
                replay=ReplayBuffer(JsonlRepository(root=data_dir)),
                run_id='selfdata', iteration=0)
trajectories = load_local_trajectories(root=data_dir)
obs, labels = build_training_pairs(trajectories, noise_std=0.02, seed=1)
print(f'self-generated trajectories: {len(trajectories)}  |  training pairs: {len(obs)}')

## 3. Fine-tune the LoRA adapter on the A100

In [ ]:
import torch
from splatforge.policy.lora_trainer import train_lora_policy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
policy, history = train_lora_policy(obs, labels, epochs=400, lr=1e-2, device=device)
print('device:', device, '| loss:', round(history[0], 5), '->', round(history[-1], 5))

trainable = sum(p.numel() for p in policy.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in policy.model.parameters())
print(f'LoRA trainable params: {trainable}/{total}')

import matplotlib.pyplot as plt
plt.plot(history, marker='o'); plt.title('LoRA behaviour-cloning loss')
plt.xlabel('checkpoint'); plt.ylabel('MSE'); plt.grid(True); plt.show()

## 4. Did it learn? Base vs LoRA grasp success
Adapter **off** = the nominal-aim baseline; **on** = the self-trained policy.

In [ ]:
from splatforge.policy.lora_trainer import evaluate_success

base = evaluate_success(policy, backend, scene, task, n=80, use_adapter=False, seed=7)
lora = evaluate_success(policy, backend, scene, task, n=80, use_adapter=True, seed=7)
print(f'base (nominal aim): {base:.1%}')
print(f'LoRA-trained:       {lora:.1%}')

import matplotlib.pyplot as plt
plt.bar(['base', 'LoRA'], [base * 100, lora * 100], color=['#999999', '#1D76DB'])
plt.ylabel('grasp success %'); plt.title('Self-generated LoRA fine-tune'); plt.show()

## 5. Save + download the adapter
Pull the adapter back into the repo under `adapters/` so the loop can load it.

In [ ]:
import shutil
from google.colab import files

policy.save('adapters/colab_a100')
shutil.make_archive('a5_lora_adapter', 'zip', 'adapters/colab_a100')
files.download('a5_lora_adapter.zip')

## Next
- Unzip into `adapters/colab_a100/` locally; `NeuralGraspPolicy.load(...)` reads it.
- To re-bank the demo curve with the trained policy, wire it into the loop's
  distill step and re-run `scripts/produce_real_curve.py` (set `GEMINI_API_KEY`
  for real curriculum + critic).